# 第 4 课：奖励函数

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>访问 <code>requirements.txt</code> 文件：</b> 1) 点击 notebook 顶部菜单中的 <em>"File"</em>，然后 2) 点击 <em>"Open"</em>。

<p> ⬇ &nbsp; <b>下载 Notebook：</b> 1) 点击 notebook 顶部菜单中的 <em>"File"</em>，然后 2) 点击 <em>"Download as"</em> 并选择 <em>"Notebook (.ipynb)"</em>。</p>

<p> 📒 &nbsp; 如需更多帮助，请参阅 <em>"Appendix – Tips, Help, and Download"</em> 课程。</p>

</div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>运行结果可能不同：</b> AI 聊天模型生成的输出具有动态性和概率性，因此每次执行的结果都可能不同。如果你的结果和视频中的展示不一致，不必惊讶。</p>

先导入依赖：

In [1]:
from utils import *
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
import torch

创建一个你将用来调用模型的 Predibase 部署：

In [ ]:
# 如果你在自己的环境中运行，请取消下面一行的注释；这里已经为你设置好了 deployment
# create_deployment()

指定要使用的模型：

In [4]:
model_id = "Qwen/Qwen2.5-7B-Instruct"

## 定义一个简单的奖励函数

In [ ]:
def wordle_reward(guess: str, secret_word: str) -> int:
    if guess.upper() == secret_word.upper():
        return 1   # 猜对了
    else:
        return 0   # 猜错了

定义一个 secret word，并获取过去猜测的反馈，然后用上面的奖励函数为这些猜测打分：

In [6]:
secret_word = "POUND"

past_guesses = [
    GuessWithFeedback.from_secret(guess="CRANE", secret=secret_word),
    GuessWithFeedback.from_secret(guess="BLOND", secret=secret_word),
    GuessWithFeedback.from_secret(guess="FOUND", secret=secret_word),
]
past_guesses

[CRANE → Feedback: C(x) R(x) A(x) N(✓) E(x),
 BLOND → Feedback: B(x) L(x) O(-) N(✓) D(✓),
 FOUND → Feedback: F(x) O(✓) U(✓) N(✓) D(✓)]

In [7]:
response = generate(get_messages(past_guesses))[0]
guess = extract_guess(response)
reward = wordle_reward(guess, secret_word)

print(f"Guessed Word: {guess} -> Reward: {reward}")

Guessed Word: DONE -> Reward: 0


## 使用奖励来计算 advantage

In [ ]:
def compute_advantages(rewards: list):
    rewards = np.array(rewards)
    
    # 计算奖励的均值和标准差
    mean_reward = np.mean(rewards)
    std_reward = np.std(rewards)

    # 避免在方差为零时出现除零错误（通常发生在所有奖励都为 0 时）
    # 注意：在 GRPO 的实现中，我们会给 std_reward 加上 1e-4，以避免除零
    if std_reward == 0:
        return [0] * len(rewards)

    # 通过除以奖励的标准差进行标准化
    advantages = (rewards - mean_reward) / std_reward
    return advantages.tolist()

In [9]:
rewards = [0.0, 0.2, 0.4, 0.5, 0.5, 0.6, 0.8, 1.0]
compute_advantages(rewards)

[-1.6903085094570331,
 -1.01418510567422,
 -0.33806170189140655,
 0.0,
 0.0,
 0.33806170189140655,
 1.0141851056742202,
 1.6903085094570331]

In [10]:
def render_guess_table(response, reward_fn):
    guesses = [extract_guess(guess) for guess in response]
    rewards = [reward_fn(guess, secret_word) for guess in guesses]
    print_guesses_table(guesses, rewards)

In [11]:
print(f"Secret: {secret_word}")
response = generate(get_messages(past_guesses), num_guesses=8)
render_guess_table(response, wordle_reward)

Secret: POUND
+---------+---------+----------+-------------+
|   Index | Guess   |   Reward |   Advantage |
+=========+=========+==========+=============+
|       0 | ROUND   |        0 |           0 |
+---------+---------+----------+-------------+
|       1 | DONE    |        0 |           0 |
+---------+---------+----------+-------------+
|       2 | NOUSE   |        0 |           0 |
+---------+---------+----------+-------------+
|       3 | NODE    |        0 |           0 |
+---------+---------+----------+-------------+
|       4 | FINDS   |        0 |           0 |
+---------+---------+----------+-------------+
|       5 | WORD    |        0 |           0 |
+---------+---------+----------+-------------+
|       6 | FLOWN   |        0 |           0 |
+---------+---------+----------+-------------+
|       7 | TROWN   |        0 |           0 |
+---------+---------+----------+-------------+


## 更新奖励函数，给予部分分数

In [ ]:
def wordle_reward_partial_credit(guess: str, secret_word: str) -> float:
    if len(guess) != len(secret_word):
        # 字母数量不对时不给奖励
        return 0.0
    
    valid_letters = set(secret_word)
    reward = 0.0
    for letter, secret_letter in zip(guess, secret_word):
        if letter == secret_letter:
            # 字母正确且位置正确
            reward += 0.2
        elif letter in valid_letters:
            # 字母正确但位置错误
            reward += 0.1
        else:
            # 不给奖励
            pass
    return reward

尝试使用更新后的奖励函数为一组响应打分。先将 <b>temperature = 0</b>。

In [13]:
print(f"Secret: {secret_word}")
response = generate(get_messages(past_guesses), num_guesses=8, temperature=0)
render_guess_table(response, wordle_reward_partial_credit)

Secret: POUND
+---------+---------+----------+-------------+
|   Index | Guess   |   Reward |   Advantage |
+=========+=========+==========+=============+
|       0 | DOWNY   |      0.5 |           0 |
+---------+---------+----------+-------------+
|       1 | DOWNY   |      0.5 |           0 |
+---------+---------+----------+-------------+
|       2 | DOWNY   |      0.5 |           0 |
+---------+---------+----------+-------------+
|       3 | DOWNY   |      0.5 |           0 |
+---------+---------+----------+-------------+
|       4 | DOWNY   |      0.5 |           0 |
+---------+---------+----------+-------------+
|       5 | DOWNY   |      0.5 |           0 |
+---------+---------+----------+-------------+
|       6 | DOWNY   |      0.5 |           0 |
+---------+---------+----------+-------------+
|       7 | DOWNY   |      0.5 |           0 |
+---------+---------+----------+-------------+


现在把 temperature 设为较高的值：

In [14]:
print(f"Secret: {secret_word}")
response = generate(get_messages(past_guesses), num_guesses=8, temperature=1.3)
render_guess_table(response, wordle_reward_partial_credit)

Secret: POUND
+---------+---------+----------+-------------+
|   Index | Guess   |   Reward |   Advantage |
+=========+=========+==========+=============+
|       0 | CONDN   |      0.5 |    1.5882   |
+---------+---------+----------+-------------+
|       1 | B BOUND |      0   |   -0.855186 |
+---------+---------+----------+-------------+
|       2 |         |      0   |   -0.855186 |
+---------+---------+----------+-------------+
|       3 | GROWN   |      0.2 |    0.122169 |
+---------+---------+----------+-------------+
|       4 | SLACK   |      0   |   -0.855186 |
+---------+---------+----------+-------------+
|       5 | MODE    |      0   |   -0.855186 |
+---------+---------+----------+-------------+
|       6 | FINDS   |      0.2 |    0.122169 |
+---------+---------+----------+-------------+
|       7 | RONDN   |      0.5 |    1.5882   |
+---------+---------+----------+-------------+


最后，把 temperature 设为适中的 0.7：

In [15]:
print(f"Secret: {secret_word}")
response = generate(get_messages(past_guesses), num_guesses=8, temperature=0.7)
render_guess_table(response, wordle_reward_partial_credit)

Secret: POUND
+---------+---------+----------+-------------+
|   Index | Guess   |   Reward |   Advantage |
+=========+=========+==========+=============+
|       0 | SCOLD   |      0.3 |    -0.11547 |
+---------+---------+----------+-------------+
|       1 | WONDS   |      0.4 |     0.34641 |
+---------+---------+----------+-------------+
|       2 | SCENT   |      0.2 |    -0.57735 |
+---------+---------+----------+-------------+
|       3 | NUDGE   |      0.3 |    -0.11547 |
+---------+---------+----------+-------------+
|       4 | SKATE   |      0   |    -1.50111 |
+---------+---------+----------+-------------+
|       5 | FLUTE   |      0.2 |    -0.57735 |
+---------+---------+----------+-------------+
|       6 | WORDN   |      0.4 |     0.34641 |
+---------+---------+----------+-------------+
|       7 | TOUND   |      0.8 |     2.19393 |
+---------+---------+----------+-------------+
